## SFT Training on No Robots with Together API

This notebook demonstrates **Supervised Fine-Tuning (SFT)** on the [HuggingFaceH4/no_robots](https://huggingface.co/datasets/HuggingFaceH4/no_robots) instruction-following dataset using the Together Post Training API.

SFT works as follows:
1. **Tokenize** each conversation (user/assistant turns) and build a loss mask that trains only on assistant tokens.
2. Run a **forward-backward** pass with cross-entropy loss.
3. Apply an **optimizer step** to update the model parameters.

Pipeline overview:

```
Create session ─► [Tokenize Batch ─► Forward/Backward ─► Optim Step] × N ─► Stop session
```


Prerequisites

Before running this notebook make sure you have:

1. Installed the `together` package (`uv sync` in this repo; the post-training helpers live in `together.post_training`).
2. Installed the `post_training_cookbook` package from this repo (`pip install -e .`).
3. Set `TOGETHER_API_KEY` and `HF_TOKEN` in a `.env` file at the repo root (or as environment variables).

In [ ]:
from __future__ import annotations

import os
import time

import datasets
import matplotlib.pyplot as plt
import wandb
from dotenv import load_dotenv
from together import AsyncTogether
from together.post_training import (
    AdamParams,
    ComputeConfig,
    LossConfig,
    ModelResourcesClient,
    Sample,
    download_checkpoint_async,
)
from transformers import AutoTokenizer, PreTrainedTokenizerBase

from post_training_cookbook.samples import build_cross_entropy_sample, is_trainable, trained_slots

load_dotenv(override=True)

In [ ]:
BASE_URL = os.environ.get("TOGETHER_BASE_URL", "https://api.together.ai/v1")
API_KEY = os.environ.get("TOGETHER_API_KEY", "")
MODEL = "Qwen/Qwen3.8-27B"
TOKENIZER_MODEL = MODEL
HF_TOKEN = os.environ.get("HF_TOKEN", "")

DATASET_NAME = "HuggingFaceH4/no_robots"
DATASET_CONFIG = None
DATASET_SPLIT = "train"

BATCH_SIZE = 8
MAX_STEPS = 20
MAX_LENGTH = 4096
LEARNING_RATE = 1e-4
TIMEOUT = 7200.0

WANDB_PROJECT = "sft-demo"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", None)

## Helper Functions

**`build_sft_sample`** (defined below) uses the tokenizer's chat template to find the boundary between the prompt and the last assistant response, then builds a loss mask that trains only on assistant tokens. The tokens and mask go to `build_cross_entropy_sample` (from `post_training_cookbook.samples`), which returns the target-aligned training `Sample`. N tokens train N-1 slots, and the builder trims the tokens and the mask together — `https://docs.together.ai/docs/post-training/sdk-reference` ("The one rule") states the layout:

### Loss type

This notebook uses cross-entropy (`LOSS_TYPE_CROSS_ENTROPY`) via `LossConfig`. Policy losses live on `GRPOConfig` / `LossConfig` — see `grpo_example.ipynb`.

In [ ]:
def build_sft_sample(
    messages: list[dict[str, str]],
    *,
    tokenizer: PreTrainedTokenizerBase,
    max_length: int,
) -> Sample | None:
    """Tokenize a conversation into a Sample that trains on the last assistant response.

    Returns None when max_length leaves no trained token.
    """
    # Encode everything up to (but not including) the last assistant message
    prefix_tokens = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)[
        "input_ids"
    ]

    # Encode the full conversation
    full_tokens = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)["input_ids"]

    # Loss mask: 0 for prompt tokens, 1 for assistant response tokens
    loss_mask = [0] * len(prefix_tokens) + [1] * (len(full_tokens) - len(prefix_tokens))

    tokens, mask = full_tokens[:max_length], loss_mask[:max_length]
    if not is_trainable(mask):
        # max_length clipped away every assistant token, so there is no slot to
        # train. The builder rejects that payload; skip the row instead.
        return None

    return build_cross_entropy_sample(tokens, mask)

## Load Dataset

[HuggingFaceH4/no_robots](https://huggingface.co/datasets/HuggingFaceH4/no_robots) is a high-quality dataset of 10K instruction-following conversations created by skilled human annotators. Each example contains multi-turn `messages` with user and assistant roles.

We use the HuggingFace tokenizer's built-in `apply_chat_template` to convert conversation messages into token IDs. This handles the model's chat format (special tokens, role markers, etc.) so conversations are encoded exactly as the model expects.

In [ ]:
if DATASET_CONFIG:
    raw_dataset = datasets.load_dataset(DATASET_NAME, DATASET_CONFIG)
else:
    raw_dataset = datasets.load_dataset(DATASET_NAME)

dataset = raw_dataset[DATASET_SPLIT].shuffle(seed=0)

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_MODEL,
    use_fast=True,
    token=HF_TOKEN or None,
)

total_batches = len(dataset) // BATCH_SIZE
total_steps = min(MAX_STEPS, total_batches)

print(f"Dataset size: {len(dataset)} examples")
print(f"Training for {total_steps} steps (batch_size={BATCH_SIZE})")
print()

for i in range(3):
    row = dataset[i]
    msgs = row["messages"]
    print(f"--- Example {i} ---")
    for msg in msgs:
        preview = msg["content"][:120].replace("\n", " ")
        print(f"  [{msg['role']}] {preview}...")
    print()

## Create Training Session

A run provisions a **model resource** (a GPU pod hosting the base model) and then attaches a **training session** to it. We use `ModelResourcesClient.create`, which provisions the resource and polls until it reaches `READY`, then `create_session` to start the session.

Since this is SFT (no rollouts needed), we pass `compute_config=ComputeConfig(num_generator_replicas=0)` so the platform provisions a **trainer-only** resource and skips spinning up a separate generator replica. Accessing `session.generator` raises `RuntimeError` on a trainer-only session.

Provisioning takes a few minutes. The resource will automatically shut down after 30 minutes of inactivity.

To resume from a training checkpoint, pass `resume_from_checkpoint_id` when re-creating the session (`create_session` / `SessionClient.create`). Omit `load_optimizer` (or pass `True`) to restore optimizer state and the step counter — pause/continue this SFT run. Pass `load_optimizer=False` to load weights only (fresh optimizer, step 0) and start a different run from those weights (for example GRPO after SFT). Weights-only resume is not valid for inference or Hugging Face checkpoints.

> **Note:** Jupyter notebooks run their own asyncio event loop, so we use the `_async` variants (`create_async`, `create_session_async`, `forward_backward_async`, etc.) with `await` throughout this notebook. If you're running a standalone Python script instead, you can use the sync methods directly (e.g. `ModelResourcesClient.create(...)`, `session.trainer.forward_backward(...)`) — no `await` needed.

> **Tip:** `ModelResourcesClient.create` can be used as a context manager to automatically stop the resource (and its sessions) when you're done:
> ```python
> with ModelResourcesClient.create(base_model=MODEL, api_key=API_KEY, compute_config=ComputeConfig(num_generator_replicas=0)) as model_resources:
>     session = model_resources.create_session()
>     result = session.trainer.forward_backward(...)
>     ...
> # resource is stopped and GPUs released on exit
> ```

In [ ]:
model_resources = await ModelResourcesClient.create_async(
    base_model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    compute_config=ComputeConfig(num_generator_replicas=0),
)
session = await model_resources.create_session_async()
print(f"Session ready: {session.session_id}")

## Training Loop

Each step of the loop:
1. **Tokenize** -- convert batch rows into target-aligned `Sample` objects carrying token IDs, next-token targets, and a loss mask selecting only assistant tokens.
2. **Forward/Backward** -- send the batch to the cross-entropy loss endpoint.
3. **Optim Step** -- apply an AdamW parameter update with linear learning-rate decay.

Per-step metrics include the **cross-entropy loss**, **current learning rate**, and **batch statistics** (sequence lengths, mask coverage).


In [ ]:
history: dict[str, list[float]] = {
    "loss": [],
    "learning_rate": [],
}

wandb_run = None
if WANDB_PROJECT:
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        config={
            "model": MODEL,
            "dataset": DATASET_NAME,
            "batch_size": BATCH_SIZE,
            "max_steps": MAX_STEPS,
            "learning_rate": LEARNING_RATE,
            "max_length": MAX_LENGTH,
        },
    )

overall_start = time.monotonic()
try:
    for step in range(total_steps):
        batch_start = step * BATCH_SIZE
        batch_end = min((step + 1) * BATCH_SIZE, len(dataset))
        batch_rows = dataset.select(range(batch_start, batch_end))

        built = (
            build_sft_sample(
                row["messages"],
                tokenizer=tokenizer,
                max_length=MAX_LENGTH,
            )
            for row in batch_rows
        )
        samples = [sample for sample in built if sample is not None]
        if not samples:
            print(f"[step {step}] every row was truncated past its response; skipping")
            continue

        lr_mult = max(0.0, 1.0 - step / max(total_steps, 1))
        current_lr = LEARNING_RATE * lr_mult

        fwd_bwd_result = await session.trainer.forward_backward_async(
            loss=LossConfig(type="LOSS_TYPE_CROSS_ENTROPY"),
            samples=samples,
            timeout=TIMEOUT,
        )
        # The service returns the total cross entropy of the batch, not a mean
        # value. Divide by the trained-slot count so steps stay comparable.
        loss_sum = fwd_bwd_result.loss
        trained = trained_slots(samples)
        loss_value = loss_sum / trained if trained else float("nan")

        await session.trainer.optim_step_async(
            adam_params=AdamParams(
                beta1=0.9,
                beta2=0.95,
                eps=1e-8,
                weight_decay=0.1,
                learning_rate=current_lr,
            ),
            timeout=TIMEOUT,
        )
        history["loss"].append(loss_value if loss_value is not None else float("nan"))
        history["learning_rate"].append(current_lr)

        if wandb_run is not None:
            wandb_run.log(
                {"loss": loss_value, "learning_rate": current_lr, "batch_size": len(samples)},
                step=step,
            )

        elapsed = time.monotonic() - overall_start
        print(
            f"[step {step}] loss={loss_value:.4f} "
            f"(loss_sum={loss_sum:.1f} over {trained:g} slots) "
            f"lr={current_lr:.6g} samples={len(samples)} elapsed={elapsed:.1f}s"
        )
finally:
    if wandb_run is not None:
        wandb.finish()
    print(f"Training completed in {time.monotonic() - overall_start:.1f}s")

## Results

Training dashboard with two panels:
- **Cross-Entropy Loss** -- mean cross entropy for each trained token, over steps.
- **Learning Rate** -- linear decay schedule from the initial rate to zero.

In [ ]:
steps = list(range(len(history["loss"])))

fig, (ax_loss, ax_lr) = plt.subplots(1, 2, figsize=(13, 4.5))

ax_loss.plot(steps, history["loss"], marker="o", markersize=3)
ax_loss.set_xlabel("Step")
ax_loss.set_ylabel("Loss")
ax_loss.set_title("Cross-Entropy Loss (token mean)")
ax_loss.grid(True, alpha=0.3)

ax_lr.plot(steps, history["learning_rate"], marker="o", markersize=3, color="tab:orange")
ax_lr.set_xlabel("Step")
ax_lr.set_ylabel("Learning Rate")
ax_lr.set_title("Learning Rate Schedule")
ax_lr.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Saving & Retrieving Training Artifacts

After training, you have several options for saving and retrieving your model:

| Operation | What it does | When to use |
| --- | --- | --- |
| **Inference checkpoint** | Saves LoRA adapter, registers as a model you can download or serve | You want to use the trained model |
| **Training checkpoint** | Saves adapter + optimizer state + step | Pause/continue the same run, or start a new objective from those weights (`load_optimizer=False`) |
| **Download checkpoint** | Downloads model files to your local machine | You want local weights (merged or adapter-only) |
| **Session query** | Lists all checkpoints saved during the session | You want to review what was saved |

In [ ]:
# -- Inference checkpoint: save the trained model for serving or download --
print("Creating inference checkpoint...")
checkpoint = await session.create_inference_checkpoint_async()
print(f"Inference checkpoint created: {checkpoint.id}")

### Deploy the checkpoint

The inference checkpoint registered its weights in the model registry. Print the ID to deploy:


In [ ]:
reg = (await session.retrieve_async()).inference_checkpoints[-1].registration
print("adapter:", reg.adapter.id if reg and reg.adapter else None)
print("merged model:", reg.model.id if reg and reg.model else None)

Then deploy it on Dedicated Model Inference (together-py 3.0+ CLI):

```bash
# Production: replicas merge the adapter into the base model at startup
tg beta endpoints deploy <adapter_id> --endpoint my-model --merge --config <config_id> --project <project_id>

# Comparison: attach several adapters to one multi-LoRA deployment
tg beta endpoints deploy <adapter_id> --endpoint my-sweep --attach-adapter
```

Guides: [Deploy a checkpoint](https://docs.together.ai/docs/post-training/deployment), [Multi-LoRA deployment](https://docs.together.ai/docs/post-training/multi-lora-deployment), [Prepare for FP4 inference](https://docs.together.ai/docs/post-training/prepare-for-fp4-inference).

In [ ]:
# -- Download inference checkpoint to your local machine --
session_info = await session.retrieve_async()
latest_ckpt = session_info.inference_checkpoints[-1]
client = AsyncTogether()
print(f"Downloading checkpoint from step {latest_ckpt.step} ({latest_ckpt.id})...")

In [ ]:
# Option A: merged model (base weights + LoRA merged)
merged_paths = await download_checkpoint_async(
    client,
    checkpoint_id=latest_ckpt.id,
    variant="CHECKPOINT_VARIANT_MERGED",
    output_dir="./my_model_merged",
)
print(f"Merged model: {len(merged_paths)} files → ./my_model_merged/")

In [ ]:
# Option B: adapter only
adapter_paths = await download_checkpoint_async(
    client,
    checkpoint_id=latest_ckpt.id,
    variant="CHECKPOINT_VARIANT_ADAPTER",
    output_dir="./my_model_adapter",
)
print(f"Adapter only: {len(adapter_paths)} files → ./my_model_adapter/")

In [ ]:
# -- Training checkpoint: save optimizer state so you can resume later --
training_ckpt = await session.create_training_checkpoint_async()
print(f"Training checkpoint: {training_ckpt.id}")
print(f'Resume by passing resume_from_checkpoint_id="{training_ckpt.id}" when re-creating the session')

### Weights-only resume (SFT → GRPO)

This notebook's resource is trainer-only (`num_generator_replicas=0`). GRPO needs a generator, so SFT→RL is a **new** resource with the same base model and the same LoRA config — not an attach to this SFT resource.

`create_session` / `SessionClient.create` take `resume_from_checkpoint_id` and `load_optimizer`. Default/`True` continues this SFT run. `load_optimizer=False` loads adapter weights only (fresh optimizer, step 0). Not valid for inference or Hugging Face checkpoints.

Cookbook recipe: `recipes/sft_then_grpo.py`. `SFTConfig` / `GRPOConfig` expose the same fields.

Do not run this on the trainer-only resource above. Same base model and LoRA config; generator replicas required:

```python
grpo_resources = await ModelResourcesClient.create_async(
    base_model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)
grpo_session = await grpo_resources.create_session_async(
    resume_from_checkpoint_id=training_ckpt.id,
    load_optimizer=False,
)
```

In [ ]:
# -- Session query: review all checkpoints saved during training --
# (Reuses the metadata fetched above; call session.retrieve_async() if running standalone.)
print(f"Session {session_info.id} — {session_info.status}")
print(f"Model resource: {session_info.resources_id}")
print(f"Current step: {session_info.step}")
print(f"\nInference checkpoints ({len(session_info.inference_checkpoints)}):")
for ckpt in session_info.inference_checkpoints:
    print(f"  Step {ckpt.step}: id={ckpt.id} registration={ckpt.registration}")
print(f"\nTraining checkpoints ({len(session_info.training_checkpoints)}):")
for ckpt in session_info.training_checkpoints:
    print(f"  Step {ckpt.step}: id={ckpt.id}")

## Cleanup

Stop the model resource to release the GPUs. Stopping the resource also ends the attached training session.

In [ ]:
await session.detach_async()
await model_resources.stop_async()
print(f"Resource {model_resources.model_resources_id} stopped.")